# Describe de la muestra estratificada

Consume los Parquet generados por [02_stratified_sampling.ipynb](02_stratified_sampling.ipynb) en `data/samples/` y corre un `describe()` sobre reviews y metadata.

In [ ]:
from pathlib import Path

import pandas as pd

SAMPLES_DIR = Path("../data/samples")

reviews_df = pd.read_parquet(SAMPLES_DIR / "reviews_sample.parquet")
meta_df = pd.read_parquet(SAMPLES_DIR / "meta_sample.parquet")

print(f"reviews: {reviews_df.shape}")
print(f"metadata: {meta_df.shape}")

In [ ]:
def split_list_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Separa columnas con listas/dicts (no hasheables) del resto.

    `describe(include="all")` cuelga sobre columnas tipo `images`/`features`/`details`, etc.,
    porque para calcular unique/top/freq compara elemento a elemento cuando el tipo no es
    hasheable. Se las excluye del describe estándar y se resumen aparte (null count, largo medio).
    """
    list_cols, plain_cols = [], []
    for col in df.columns:
        sample = df[col].dropna()
        is_list_like = len(sample) > 0 and isinstance(sample.iloc[0], (list, dict))
        (list_cols if is_list_like else plain_cols).append(col)
    return list_cols, plain_cols


def summarize_list_columns(df: pd.DataFrame, list_cols: list[str]) -> pd.DataFrame:
    def length(value):
        if value is None:
            return None
        try:
            return len(value)
        except TypeError:
            return None

    return pd.DataFrame(
        {
            "non_null": [df[c].notna().sum() for c in list_cols],
            "avg_len": [df[c].map(length).mean() for c in list_cols],
        },
        index=list_cols,
    )

## Reviews

In [ ]:
reviews_list_cols, reviews_plain_cols = split_list_columns(reviews_df)
reviews_df[reviews_plain_cols].describe(include="all").T

In [ ]:
summarize_list_columns(reviews_df, reviews_list_cols)

## Metadata

In [ ]:
meta_list_cols, meta_plain_cols = split_list_columns(meta_df)
meta_df[meta_plain_cols].describe(include="all").T

In [ ]:
summarize_list_columns(meta_df, meta_list_cols)